# Notebook 3: Gold Aggregations
This notebook represents the Gold aggregation stage for the Automated Data Guard batch pipeline.

Purpose:
* Read the trusted Silver Delta dataset produced by Notebook 2
* Build 5 Gold aggregation tables for business analytics
* Write each aggregation as a separate Delta table in the Gold ADLS container
* Return an ADF-friendly JSON payload with run status, output paths, and row counts

Architecture position:
* Upstream: Silver validated Delta output from Notebook 2 DQ and Split
* Current stage: Gold aggregations
* Downstream: ADF orchestration, reporting, dashboards

## Inputs and Outputs

Expected inputs:
* `silver_path`: Silver Delta location in ADLS (passed from ADF or widget default)
* `gold_base_path`: root Gold container path in ADLS
* `ingestion_date`: partition date used for Gold output sub-paths
* Secret-backed ADLS access via `dq-secret-scope`

Expected outputs:
* Gold Delta: accidents by state -> `gold_base_path/by_state/ingestion_date=...`
* Gold Delta: accidents by severity -> `gold_base_path/by_severity/ingestion_date=...`
* Gold Delta: accidents by hour of day -> `gold_base_path/by_hour/ingestion_date=...`
* Gold Delta: accidents by weather condition -> `gold_base_path/by_weather_condition/ingestion_date=...`
* Gold Delta: accidents daily trend -> `gold_base_path/daily_trend/ingestion_date=...`
* ADF return payload with row counts and run status

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
import json

print("Notebook 3 libraries loaded.")

Notebook 3 libraries loaded.


In [0]:
storage_account_name = "stdqprojectmouni"
default_ingestion_date = "2026-05-24"
default_silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/us_accidents/validated/ingestion_date={default_ingestion_date}"
default_gold_base_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/us_accidents"

def get_widget_value(widget_name, default_value):
    try:
        value = dbutils.widgets.get(widget_name)
        return value if value not in (None, "") else default_value
    except Exception:
        return default_value

try:
    dbutils.widgets.text("silver_path", default_silver_path, "silver_path")
    dbutils.widgets.text("gold_base_path", default_gold_base_path, "gold_base_path")
    dbutils.widgets.text("ingestion_date", default_ingestion_date, "ingestion_date")
except Exception:
    pass

silver_path = get_widget_value("silver_path", default_silver_path)
gold_base_path = get_widget_value("gold_base_path", default_gold_base_path)
ingestion_date = get_widget_value("ingestion_date", default_ingestion_date)

gold_by_state_path       = f"{gold_base_path}/by_state/ingestion_date={ingestion_date}"
gold_by_severity_path    = f"{gold_base_path}/by_severity/ingestion_date={ingestion_date}"
gold_by_hour_path        = f"{gold_base_path}/by_hour/ingestion_date={ingestion_date}"
gold_by_weather_path     = f"{gold_base_path}/by_weather_condition/ingestion_date={ingestion_date}"
gold_daily_trend_path    = f"{gold_base_path}/daily_trend/ingestion_date={ingestion_date}"

storage_access_ready = False
storage_access_message = "Storage configuration not attempted yet."

try:
    storage_account_key = dbutils.secrets.get(scope="dq-secret-scope", key="storage-account-key")
    spark.conf.set(
        f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
        storage_account_key
    )
    storage_access_ready = True
    storage_access_message = "Storage access configured from secret scope."
except Exception as e:
    storage_access_message = (
        f"Storage access configuration failed. Use dq-cluster. Root cause: {str(e)}"
    )

print("Notebook 3 runtime configuration set.")
print("Silver path       :", silver_path)
print("Gold base path    :", gold_base_path)
print("Ingestion date    :", ingestion_date)
print("By state path     :", gold_by_state_path)
print("By severity path  :", gold_by_severity_path)
print("By hour path      :", gold_by_hour_path)
print("By weather path   :", gold_by_weather_path)
print("Daily trend path  :", gold_daily_trend_path)
print("Storage ready     :", storage_access_ready)
print("Storage status    :", storage_access_message)

Notebook 3 runtime configuration set.
Silver path       : abfss://silver@stdqprojectmouni.dfs.core.windows.net/us_accidents/validated/ingestion_date=2026-05-24
Gold base path    : abfss://gold@stdqprojectmouni.dfs.core.windows.net/us_accidents
Ingestion date    : 2026-05-24
By state path     : abfss://gold@stdqprojectmouni.dfs.core.windows.net/us_accidents/by_state/ingestion_date=2026-05-24
By severity path  : abfss://gold@stdqprojectmouni.dfs.core.windows.net/us_accidents/by_severity/ingestion_date=2026-05-24
By hour path      : abfss://gold@stdqprojectmouni.dfs.core.windows.net/us_accidents/by_hour/ingestion_date=2026-05-24
By weather path   : abfss://gold@stdqprojectmouni.dfs.core.windows.net/us_accidents/by_weather_condition/ingestion_date=2026-05-24
Daily trend path  : abfss://gold@stdqprojectmouni.dfs.core.windows.net/us_accidents/daily_trend/ingestion_date=2026-05-24
Storage ready     : True
Storage status    : Storage access configured from secret scope.


In [0]:
silver_df = None

if not storage_access_ready:
    print("Silver read skipped because storage access is not ready.")
    print(storage_access_message)
else:
    try:
        silver_df = spark.read.format("delta").load(silver_path)
        silver_df.createOrReplaceTempView("silver_table")
        total_silver_rows = silver_df.count()
        print("Silver Delta loaded successfully.")
        print("Total Silver rows:", total_silver_rows)
        display(silver_df.limit(5))
    except Exception as e:
        print("Silver read failed:", str(e))

Silver Delta loaded successfully.
Total Silver rows: 7693711


id,source,severity,start_time,end_time,start_lat,start_lng,end_lat,end_lng,distance_mi,description,street,city,county,state,zipcode,country,timezone,airport_code,weather_timestamp,temperature_f,wind_chill_f,humidity,pressure_in,visibility_mi,wind_direction,wind_speed_mph,precipitation_in,weather_condition,amenity,bump,crossing,give_way,junction,no_exit,railway,roundabout,station,stop,traffic_calming,traffic_signal,turning_loop,sunrise_sunset,civil_twilight,nautical_twilight,astronomical_twilight,id_present_rule,severity_rule,start_time_rule,start_lat_rule,start_lng_rule,state_rule,country_rule,end_time_rule,zipcode_rule,street_rule,city_rule,county_rule,timezone_rule,humidity_rule,visibility_rule,pressure_rule,temperature_rule,weather_timestamp_rule,duplicate_id_rule,dq_reason,core_rule_pass,optional_quality_flag
A-3,Source2,2,2016-02-08T06:49:27Z,2016-02-08T07:19:27Z,39.063148,-84.032608,null,null,0.01,Accident on OH-32 State Route 32 Westbound at Dela Palma Rd. Expect delays.,State Route 32,Williamsburg,Clermont,OH,45176,US,US/Eastern,KI69,2016-02-08T06:56:00Z,36.0,33.3,100.0,29.67,10.0,SW,3.5,null,Overcast,false,false,false,false,false,false,false,false,false,false,false,true,false,Night,Night,Day,Day,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,,true,true
A-227,Source2,2,2016-02-17T10:00:29Z,2016-02-17T10:30:29Z,39.792931,-84.215195,null,null,0.01,Accident on OH-48 Main St at Pointview Ave. Expect delays.,N Main St,Dayton,Montgomery,OH,45405-2858,US,US/Eastern,KDAY,2016-02-17T09:56:00Z,32.0,26.4,88.0,30.2,4.0,West,5.8,null,Overcast,false,false,false,false,false,false,false,false,false,false,false,false,false,Day,Day,Day,Day,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,,true,true
A-231,Source2,2,2016-02-17T12:57:34Z,2016-02-17T13:27:34Z,39.615208,-84.22731800000003,null,null,0.01,Accident on OH-741 Springboro Pike at Ferndown Dr.,Ferndown Dr,Miamisburg,Montgomery,OH,45342,US,US/Eastern,KMGY,2016-02-17T12:53:00Z,35.1,32.2,67.0,30.23,9.0,NW,3.5,null,Overcast,false,false,true,false,false,false,false,false,false,false,false,true,false,Day,Day,Day,Day,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,,true,true
A-609,Source2,2,2016-03-11T09:12:17Z,2016-03-11T09:42:17Z,39.774738,-84.174683,null,null,0.01,Accident on OH-202 Troy St at Chapel St.,Chapel St,Dayton,Montgomery,OH,45404-1809,US,US/Eastern,KFFO,2016-03-11T08:58:00Z,45.9,40.8,95.0,30.34,10.0,NNE,10.4,null,Overcast,false,false,false,false,false,false,false,false,false,false,false,false,false,Day,Day,Day,Day,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,,true,true
A-929,Source2,2,2016-06-22T18:06:34Z,2016-06-22T18:51:34Z,38.929756,-121.08866100000002,null,null,0.0,Accident on CA-49 Grass Valley Hwy Northbound at Hulbert Way.,Hulbert Way,Auburn,Placer,CA,95603,US,US/Pacific,KAUN,2016-06-22T18:15:00Z,91.4,null,17.0,29.95,10.0,South,8.1,null,Clear,false,false,false,false,false,false,false,false,false,false,false,true,false,Day,Day,Day,Day,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,,true,true


In [0]:
%sql
-- Gold Aggregation 1: Accidents by State
-- Reads from silver_table temp view created in the previous cell
-- Groups all validated accident records by US state code
-- Orders from highest accident count to lowest

SELECT
    state,
    COUNT(*) AS accident_count
FROM silver_table
GROUP BY state
ORDER BY accident_count DESC

state,accident_count
CA,1731689
FL,876914
TX,581386
SC,378213
NY,346765
NC,337301
VA,301779
PA,295725
MN,191661
OR,177640


In [0]:
%sql
-- Gold Aggregation 2: Accidents by Severity
-- Severity levels: 1 = least impact, 4 = most severe
-- Also calculates percentage share of each severity level

SELECT
    severity,
    COUNT(*) AS accident_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
FROM silver_table
GROUP BY severity
ORDER BY severity

severity,accident_count,pct_of_total
1,67001,0.87
2,6127672,79.65
3,1295599,16.84
4,203439,2.64


In [0]:
%sql
-- Gold Aggregation 3: Accidents by Hour of Day
-- Extracts the hour from start_time (0 = midnight, 12 = noon, 23 = 11pm)
-- Identifies peak accident hours across the full dataset

SELECT
    HOUR(start_time) AS hour_of_day,
    COUNT(*) AS accident_count
FROM silver_table
WHERE start_time IS NOT NULL
GROUP BY HOUR(start_time)
ORDER BY hour_of_day

hour_of_day,accident_count
0,108349
1,94986
2,92038
3,82979
4,158719
5,226879
6,404009
7,585219
8,575545
9,361809


In [0]:
%sql
-- Gold Aggregation 4: Accidents by Weather Condition
-- Groups by weather_condition column
-- Rows with null weather_condition are labeled as Unknown
-- Orders from highest to lowest accident count

SELECT
    COALESCE(weather_condition, 'Unknown') AS weather_condition,
    COUNT(*) AS accident_count
FROM silver_table
GROUP BY weather_condition
ORDER BY accident_count DESC

weather_condition,accident_count
Fair,2549472
Mostly Cloudy,1013032
Cloudy,813468
Clear,807370
Partly Cloudy,696967
Overcast,382309
Light Rain,351989
Scattered Clouds,204638
Unknown,164828
Light Snow,128186


In [0]:
%sql
-- Gold Aggregation 5: Daily Accident Trend
-- Extracts date from start_time to count accidents per calendar day
-- Useful for trend analysis and identifying high-accident days

SELECT
    TO_DATE(start_time) AS accident_date,
    COUNT(*) AS accident_count
FROM silver_table
WHERE start_time IS NOT NULL
GROUP BY TO_DATE(start_time)
ORDER BY accident_date

accident_date,accident_count
2016-01-14,7
2016-02-08,60
2016-02-09,59
2016-02-10,49
2016-02-11,93
2016-02-12,18
2016-02-13,14
2016-02-14,13
2016-02-15,62
2016-02-16,84


In [0]:
gold_write_status = "NOT_ATTEMPTED"
by_state_df = None
by_severity_df = None
by_hour_df = None
by_weather_df = None
daily_trend_df = None

if silver_df is None:
    gold_write_status = "SKIPPED_NO_SILVER"
    print("Gold writes skipped because Silver data is not available.")
else:
    try:
        # --- Agg 1: By State ---
        by_state_df = spark.sql("""
            SELECT state, COUNT(*) AS accident_count
            FROM silver_table
            GROUP BY state
            ORDER BY accident_count DESC
        """)
        by_state_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_by_state_path)
        print("Written: by_state ->", gold_by_state_path)

        # --- Agg 2: By Severity ---
        by_severity_df = spark.sql("""
            SELECT severity, COUNT(*) AS accident_count,
            ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
            FROM silver_table
            GROUP BY severity
            ORDER BY severity
        """)
        by_severity_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_by_severity_path)
        print("Written: by_severity ->", gold_by_severity_path)

        # --- Agg 3: By Hour ---
        by_hour_df = spark.sql("""
            SELECT HOUR(start_time) AS hour_of_day, COUNT(*) AS accident_count
            FROM silver_table
            WHERE start_time IS NOT NULL
            GROUP BY HOUR(start_time)
            ORDER BY hour_of_day
        """)
        by_hour_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_by_hour_path)
        print("Written: by_hour ->", gold_by_hour_path)

        # --- Agg 4: By Weather Condition ---
        by_weather_df = spark.sql("""
            SELECT COALESCE(weather_condition, 'Unknown') AS weather_condition,
            COUNT(*) AS accident_count
            FROM silver_table
            GROUP BY weather_condition
            ORDER BY accident_count DESC
        """)
        by_weather_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_by_weather_path)
        print("Written: by_weather_condition ->", gold_by_weather_path)

        # --- Agg 5: Daily Trend ---
        daily_trend_df = spark.sql("""
            SELECT TO_DATE(start_time) AS accident_date, COUNT(*) AS accident_count
            FROM silver_table
            WHERE start_time IS NOT NULL
            GROUP BY TO_DATE(start_time)
            ORDER BY accident_date
        """)
        daily_trend_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_daily_trend_path)
        print("Written: daily_trend ->", gold_daily_trend_path)

        gold_write_status = "SUCCESS"
        print("\nAll 5 Gold tables written successfully.")

    except Exception as e:
        gold_write_status = "FAILED"
        print("Gold write failed:", str(e))

print("\nGold write status:", gold_write_status)

Written: by_state -> abfss://gold@stdqprojectmouni.dfs.core.windows.net/us_accidents/by_state/ingestion_date=2026-05-24
Written: by_severity -> abfss://gold@stdqprojectmouni.dfs.core.windows.net/us_accidents/by_severity/ingestion_date=2026-05-24
Written: by_hour -> abfss://gold@stdqprojectmouni.dfs.core.windows.net/us_accidents/by_hour/ingestion_date=2026-05-24
Written: by_weather_condition -> abfss://gold@stdqprojectmouni.dfs.core.windows.net/us_accidents/by_weather_condition/ingestion_date=2026-05-24
Written: daily_trend -> abfss://gold@stdqprojectmouni.dfs.core.windows.net/us_accidents/daily_trend/ingestion_date=2026-05-24

All 5 Gold tables written successfully.

Gold write status: SUCCESS


In [0]:
gold_metrics = {}

if silver_df is None or gold_write_status != "SUCCESS":
    gold_metrics = {
        "run_status": "SKIPPED_NO_SILVER" if silver_df is None else gold_write_status,
        "by_state_rows": None,
        "by_severity_rows": None,
        "by_hour_rows": None,
        "by_weather_condition_rows": None,
        "daily_trend_rows": None,
    }
    print("Gold metrics skipped. Status:", gold_metrics["run_status"])
else:
    gold_metrics = {
        "run_status": "SUCCESS",
        "by_state_rows": by_state_df.count(),
        "by_severity_rows": by_severity_df.count(),
        "by_hour_rows": by_hour_df.count(),
        "by_weather_condition_rows": by_weather_df.count(),
        "daily_trend_rows": daily_trend_df.count(),
    }

    print("=== Gold Summary Metrics ===")
    for key, value in gold_metrics.items():
        print(f"  {key}: {value}")

    print("\n--- By State (top 10) ---")
    display(by_state_df.limit(10))

    print("\n--- By Severity ---")
    display(by_severity_df)

    print("\n--- By Hour of Day ---")
    display(by_hour_df)

    print("\n--- By Weather Condition (top 10) ---")
    display(by_weather_df.limit(10))

    print("\n--- Daily Trend (first 10 days) ---")
    display(daily_trend_df.limit(10))

=== Gold Summary Metrics ===
  run_status: SUCCESS
  by_state_rows: 49
  by_severity_rows: 4
  by_hour_rows: 24
  by_weather_condition_rows: 145
  daily_trend_rows: 2572

--- By State (top 10) ---


state,accident_count
CA,1731689
FL,876914
TX,581386
SC,378213
NY,346765
NC,337301
VA,301779
PA,295725
MN,191661
OR,177640



--- By Severity ---


severity,accident_count,pct_of_total
1,67001,0.87
2,6127672,79.65
3,1295599,16.84
4,203439,2.64



--- By Hour of Day ---


hour_of_day,accident_count
0,108349
1,94986
2,92038
3,82979
4,158719
5,226879
6,404009
7,585219
8,575545
9,361809



--- By Weather Condition (top 10) ---


weather_condition,accident_count
Fair,2549472
Mostly Cloudy,1013032
Cloudy,813468
Clear,807370
Partly Cloudy,696967
Overcast,382309
Light Rain,351989
Scattered Clouds,204638
Unknown,164828
Light Snow,128186



--- Daily Trend (first 10 days) ---


accident_date,accident_count
2016-01-14,7
2016-02-08,60
2016-02-09,59
2016-02-10,49
2016-02-11,93
2016-02-12,18
2016-02-13,14
2016-02-14,13
2016-02-15,62
2016-02-16,84


In [0]:
final_adf_payload = {
    "silver_path":                 silver_path               if "silver_path"               in globals() else None,
    "gold_base_path":              gold_base_path            if "gold_base_path"            in globals() else None,
    "ingestion_date":              ingestion_date            if "ingestion_date"            in globals() else None,
    "gold_by_state_path":          gold_by_state_path        if "gold_by_state_path"        in globals() else None,
    "gold_by_severity_path":       gold_by_severity_path     if "gold_by_severity_path"     in globals() else None,
    "gold_by_hour_path":           gold_by_hour_path         if "gold_by_hour_path"         in globals() else None,
    "gold_by_weather_path":        gold_by_weather_path      if "gold_by_weather_path"      in globals() else None,
    "gold_daily_trend_path":       gold_daily_trend_path     if "gold_daily_trend_path"     in globals() else None,
    "run_status":                  gold_metrics.get("run_status")                  if "gold_metrics" in globals() else "SKIPPED_NO_METRICS",
    "by_state_rows":               gold_metrics.get("by_state_rows")               if "gold_metrics" in globals() else None,
    "by_severity_rows":            gold_metrics.get("by_severity_rows")            if "gold_metrics" in globals() else None,
    "by_hour_rows":                gold_metrics.get("by_hour_rows")                if "gold_metrics" in globals() else None,
    "by_weather_condition_rows":   gold_metrics.get("by_weather_condition_rows")   if "gold_metrics" in globals() else None,
    "daily_trend_rows":            gold_metrics.get("daily_trend_rows")            if "gold_metrics" in globals() else None,
    "storage_access_ready":        storage_access_ready      if "storage_access_ready"      in globals() else None,
    "storage_access_message":      storage_access_message    if "storage_access_message"    in globals() else None,
}

final_adf_payload_json = json.dumps(final_adf_payload)
print("Final ADF return payload:")
print(final_adf_payload_json)

try:
    dbutils.notebook.exit(final_adf_payload_json)
except Exception as e:
    print("Notebook exit skipped in interactive mode.")
    print(f"Exit reason: {str(e)}")